[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronGrace978/Automata/blob/main/notebooks/diatom_nca_colab.ipynb)

# 🦠 Diatom NCA — Morphogenesis in Your Browser

**One shared local rule → a glass diatom grows from a single cell. Cut it, it heals. Play sound, it bends.**

Run top to bottom (Runtime → Run all). A GPU helps but everything runs on CPU too.

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────
import sys, subprocess
try:
    import torch
    print('torch', torch.__version__)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch', 'matplotlib', 'scipy', 'pillow'], check=True)
    import torch

# Pull the repo so `nca/` is importable (no-op if already present)
import os
if os.path.isdir('/content/Automata'):
    os.chdir('/content/Automata')
    !git pull -q
else:
    try:
        !git clone -q https://github.com/AaronGrace978/Automata.git /content/Automata
        os.chdir('/content/Automata')
    except Exception as e:
        print('clone skipped (running locally?):', e)
if '/content/Automata' not in sys.path:
    sys.path.insert(0, '/content/Automata')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

from nca.model import DiatomNCA
from nca import diatom as diatom_lib, audio as audio_lib
from nca.train import TrainConfig, train
from nca.utils import state_to_rgb, circle_damage, half_damage
torch.manual_seed(0); np.random.seed(0)

## 1. The Diatom road — procedural frustule targets

Centric (radial) and pennate (bilateral) shells. These are what the creature learns to grow.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes[0]):
    ax.imshow(diatom_lib.centric_diatom(size=64, n_fold=6 + i, n_rings=2 + (i % 3), seed=i * 13))
    ax.set_title(f'centric · fold={6 + i}'); ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(diatom_lib.pennate_diatom(size=64, aspect=1.8 + 0.3 * i, seed=100 + i * 7))
    ax.set_title(f'pennate · aspect={1.8 + 0.3 * i:.1f}'); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. The creature — one local rule, 16 channels per cell

`0:2` RGB · `3` alpha · `4:7` DNA genome · `8:10` morphogen · `11` AUDIO sense · `12:15` hidden/silica. Watch the untrained soup twitch:

In [ ]:
SIZE = 48
model = DiatomNCA(num_channels=16, hidden_dim=128).to(device)
with torch.no_grad():
    traj = model.grow(32, size=SIZE, device=device)  # (B,T,C,H,W)
frames = [state_to_rgb(s) for s in traj[0, ::4]]
fig, axes = plt.subplots(1, len(frames), figsize=(14, 2.5))
for ax, f in zip(axes, frames):
    ax.imshow(f); ax.axis('off')
plt.suptitle('untrained growth: t=0 → t=32 (primordial soup)'); plt.show()

## 3. Train it — pool + damage (regeneration is learned, not luck)

Short run so it finishes in minutes. For the real creature, raise `steps` to 3000+.

In [ ]:
cfg = TrainConfig(size=SIZE, hidden_dim=128, steps=400, batch=8,
                   pool_size=64, rollout_min=32, rollout_max=64,
                   damage_prob=0.5, audio_prob=0.5, log_every=100, device=device)
losses = []
def on_log(it, loss, sample):
    losses.append(loss); print(f'[{it}/{cfg.steps}] loss={loss:.5f}', flush=True)

train(model, cfg, on_log=on_log)
plt.plot(losses, marker='o'); plt.title('training loss'); plt.xlabel('log step'); plt.show()

torch.save({'state_dict': model.state_dict()}, 'diatom_nca.pt')
print('saved diatom_nca.pt — download it from the file browser')

## 4. Growth timelapse — the trained diatom assembling itself

In [ ]:
with torch.no_grad():
    traj = model.grow(96, size=SIZE, device=device)
frames = [state_to_rgb(s) for s in traj[0]]
fig = plt.figure(figsize=(4, 4))
im = plt.imshow(frames[0]); plt.axis('off'); plt.title('grown from a single cell')
HTML(animation.FuncAnimation(fig, lambda i: im.set_data(frames[i]),
     frames=len(frames), interval=60).to_jshtml())

## 5. Surgery — cut it in half. It heals.

In [ ]:
with torch.no_grad():
    x = model.seed(1, SIZE).to(device)
    x = model(x, steps=64)
    grown = state_to_rgb(x)[0]
    x = half_damage(x)
    cut = state_to_rgb(x)[0]
    x = model(x, steps=64)
    healed = state_to_rgb(x)[0]
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, img, t in zip(axes, [grown, cut, healed], ['grown', '✂️ cut', 'healed ✨']):
    ax.imshow(img); ax.set_title(t); ax.axis('off')
plt.show()

## 6. Sound bends the rule — synth beat drives morphology

8-dim audio vector → FiLM gains retune every neuron + energy scales growth rate. Swap in your own audio by uploading a WAV and using `audio_lib.load_wav()`.

In [ ]:
# from google.colab import files  # ← uncomment to upload your own WAV
# wav_path = list(files.upload().keys())[0]
# wave, sr = audio_lib.load_wav(wav_path)
wave = audio_lib.synth_demo_signal('beat', seconds=4.0)
feats = audio_lib.smooth(audio_lib.stft_features(wave))
print('audio frames:', feats.shape)

STEPS = 96
idx = (np.linspace(0, len(feats) - 1, STEPS)).astype(int)
with torch.no_grad():
    x = model.seed(1, SIZE).to(device)
    seq = []
    for t in range(STEPS):
        a = torch.from_numpy(feats[idx[t]]).unsqueeze(0).to(device)
        energy = float(a[0, 0])
        x = model.update(x, audio=a, fire_rate=0.3 + 0.6 * min(1.0, energy * 3))
        seq.append(state_to_rgb(x)[0])
fig, axes = plt.subplots(1, 5, figsize=(12, 2.8))
for ax, f in zip(axes, [seq[i] for i in np.linspace(0, STEPS - 1, 5, dtype=int)]):
    ax.imshow(f); ax.axis('off')
plt.suptitle('same genome, grown under a beat'); plt.show()

fig = plt.figure(figsize=(4, 4))
im = plt.imshow(seq[0]); plt.axis('off')
HTML(animation.FuncAnimation(fig, lambda i: im.set_data(seq[i]), frames=len(seq), interval=60).to_jshtml())

## 7. It talks — the creature sings its growth

Mass becomes pentatonic pitch, pigment becomes chord, growth becomes loudness. Listen, then play the song back and watch it grow under its own voice.

In [ ]:
from nca.voice import sing
from IPython.display import Audio

with torch.no_grad():
    song_traj = model.grow(96, size=SIZE, device=device)
song = sing(song_traj[0])
print(f'song: {len(song)/22050:.1f}s stereo')
Audio(song.T, rate=22050)

plt.figure(figsize=(10, 2))
plt.plot(song[::50, 0], alpha=0.7, label='left')
plt.plot(song[::50, 1], alpha=0.7, label='right')
plt.title("the creature's voice"); plt.legend(); plt.show()

# Self-listening: grow a twin under the song
feats2 = audio_lib.smooth(audio_lib.stft_features(song.mean(axis=1).astype(np.float32)))
idx2 = (np.linspace(0, len(feats2) - 1, 64)).astype(int)
with torch.no_grad():
    x = model.seed(1, SIZE).to(device)
    for t in range(64):
        a = torch.from_numpy(feats2[idx2[t]]).unsqueeze(0).to(device)
        x = model.update(x, audio=a)
    twin = state_to_rgb(x)[0]
plt.imshow(twin); plt.axis('off'); plt.title('grown under its own song'); plt.show()

## 8. New species — mutate the DNA channels

Same rule, different genome → different frustule.

In [ ]:
with torch.no_grad():
    genomes = torch.randn(6, 4) * 0.5
    species = []
    for g in genomes:
        x = model.seed(1, SIZE, device=device, genome=g.unsqueeze(0))
        species.append(state_to_rgb(model(x, steps=96))[0])
fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
for ax, s in zip(axes, species):
    ax.imshow(s); ax.axis('off')
plt.suptitle('one rule · six genomes · six diatoms'); plt.show()

## 9. It speaks — eyes, instinct, backbone, persona

The talking-NCA stack: the visual module reads the body, the RID instinct layer feels it, a language backbone narrates it, a persona styles it — and the words feed back into growth. Three personas, one wound.

In [ ]:
from nca.mind import DiatomMind
from nca.utils import half_damage

for persona in ['diatom_elder', 'lab_assistant', 'feral_bloom']:
    mind = DiatomMind(model, persona=persona, seed=0)
    out = mind.run(48, size=SIZE, speak_every=24)
    print(f'── {persona} ──')
    for line in out['transcript']:
        print(f"[t={line['step']}] {line['said']}")
    with torch.no_grad():
        x = half_damage(out['final'])
        print('[wounded]', mind.utter(mind.observe(x)))
    print()

## Next dives

- Train longer: `steps=3000`, `size=64` (turn up Runtime → GPU first).
- `scripts/export_weights.py` → drop `weights.json` next to `web/demo.html` for the live mic creature.
- Song → species: average a track's features into one vector, inject it as the genome, grow the album cover.